# Roostoo Strategy Backtester

Interactive backtesting notebook powered by backtrader. Fetch data once, then iterate on strategies by re-running cells.

**Workflow:**
1. Run Cell 1-2 once (setup + fetch data)
2. Tweak strategy params in Cell 3 and re-run Cells 3-5 to iterate
3. Use Cell 6 for parameter sweeps
4. Swap strategy class in Cell 7 to test a different strategy

In [1]:
# Cell 1: Setup
import sys
sys.path.insert(0, '..')  # so imports work from notebooks/

import warnings
warnings.filterwarnings('ignore')

import backtrader as bt
import pandas as pd
import yaml
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.figsize'] = (16, 10)

from backtest.bt_strategy import RoostooStrategy, build_cerebro, extract_metrics
from backtest.data_feed import fetch_binance_feed, fetch_fear_greed
from strategy.multi_signal import MultiSignalStrategy

# Load config defaults
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

PAIRS = config['trading']['pairs']
STRATEGY_CFG = config['strategy']
print(f'Pairs: {PAIRS}')
print(f'Strategy config: {STRATEGY_CFG}')

Pairs: ['BTC/USD', 'ETH/USD', 'SOL/USD']
Strategy config: {'rsi_period': 14, 'rsi_oversold': 35, 'rsi_overbought': 65, 'ema_fast': 12, 'ema_slow': 26, 'momentum_threshold_pct': 2.0, 'min_signal_score': 2}


In [2]:
# Cell 2: Fetch data (run once, reuse across backtests)
DAYS = 30

# Fetch OHLCV from Binance
raw_feeds = {}
for pair in PAIRS:
    raw_feeds[pair] = fetch_binance_feed(pair, days=DAYS)

# Fetch Fear & Greed history
fng = fetch_fear_greed(days=DAYS)

print(f'\nData ready: {len(raw_feeds)} pairs, {DAYS} days, {len(fng)} F&G entries')


Data ready: 3 pairs, 30 days, 35 F&G entries


In [3]:
# Cell 3: Run backtest
# ---- TWEAK THESE AND RE-RUN ----
strategy_config = {
    'rsi_period': 14,
    'rsi_oversold': 35,
    'rsi_overbought': 65,
    'ema_fast': 12,
    'ema_slow': 26,
    'momentum_threshold_pct': 2.0,
    'min_signal_score': 2,
}
# --------------------------------

# Need to re-create feeds each run (backtrader consumes them)
feeds = {}
for pair in PAIRS:
    feeds[pair] = fetch_binance_feed(pair, days=DAYS)

cerebro = build_cerebro(
    feeds=feeds,
    strategy_class=MultiSignalStrategy,
    strategy_config=strategy_config,
    fear_greed_map=fng,
    cash=50_000.0,
    commission=0.0005,
    stop_loss_pct=0.05,
    max_position_pct=0.20,
    kill_switch_drawdown=0.15,
)

print(f'Starting Portfolio: ${cerebro.broker.getvalue():,.2f}')
results = cerebro.run()
strat = results[0]
print(f'Final Portfolio:    ${cerebro.broker.getvalue():,.2f}')

Starting Portfolio: $50,000.00
Final Portfolio:    $51,855.68


In [4]:
# Cell 4: Plot charts (candlestick + indicators + buy/sell markers + equity curve)
cerebro.plot(
    style='candlestick',
    volume=False,
    barup='green',
    bardown='red',
)

<IPython.core.display.Javascript object>

[[<Figure size 1600x1000 with 8 Axes>]]

In [5]:
# Cell 5: Metrics tearsheet
metrics = extract_metrics(strat)

print('=' * 55)
print('  BACKTEST RESULTS')
print('=' * 55)
print(f"  Period:          {DAYS} days")
print(f"  Pairs:           {', '.join(PAIRS)}")
print(f"  Total Return:    {metrics['total_return']:>12.2%}")
print(f"  Total Trades:    {metrics['total_trades']:>12d}")
if metrics['win_rate'] is not None:
    print(f"  Win Rate:        {metrics['win_rate']:>12.1%}")
    print(f"  Won/Lost:        {metrics['won']:>5d} / {metrics['lost']}")
else:
    print(f"  Win Rate:        {'N/A':>12s}")
if metrics['avg_pnl']:
    print(f"  Avg P&L/Trade:   ${metrics['avg_pnl']:>11.2f}")
print('-' * 55)
print(f"  Sharpe Ratio:    {metrics['sharpe'] or 'N/A':>12}")
print(f"  Max Drawdown:    {metrics['max_drawdown']:>12.2%}")
print(f"  Max DD Duration: {metrics['max_dd_duration']:>12d} bars")
print(f"  SQN:             {metrics['sqn'] or 'N/A':>12}")
print('=' * 55)

# Also show as DataFrame for easy comparison
pd.DataFrame([metrics]).T.rename(columns={0: 'Value'})

  BACKTEST RESULTS
  Period:          30 days
  Pairs:           BTC/USD, ETH/USD, SOL/USD
  Total Return:           3.64%
  Total Trades:               4
  Win Rate:               25.0%
  Won/Lost:            1 / 2
  Avg P&L/Trade:   $     -18.31
-------------------------------------------------------
  Sharpe Ratio:             N/A
  Max Drawdown:           2.57%
  Max DD Duration:          403 bars
  SQN:             -0.0902528535796545


,Value
total_return,0.036442
sharpe,None
max_drawdown,0.025713
max_dd_duration,403
total_trades,4
won,1
lost,2
win_rate,0.25
sqn,-0.090253
avg_pnl,-18.310012


In [6]:
# Cell 6: Parameter sweep
# Sweep a single parameter across multiple values and compare
SWEEP_PARAM = 'rsi_oversold'
SWEEP_VALUES = [25, 30, 35, 40]

sweep_results = []
for val in SWEEP_VALUES:
    cfg = dict(strategy_config)
    cfg[SWEEP_PARAM] = val

    # Re-fetch feeds (backtrader consumes them)
    sweep_feeds = {pair: fetch_binance_feed(pair, days=DAYS) for pair in PAIRS}

    c = build_cerebro(
        feeds=sweep_feeds,
        strategy_class=MultiSignalStrategy,
        strategy_config=cfg,
        fear_greed_map=fng,
    )
    r = c.run()
    m = extract_metrics(r[0])
    m[SWEEP_PARAM] = val
    m['final_value'] = c.broker.getvalue()
    sweep_results.append(m)

sweep_df = pd.DataFrame(sweep_results)
sweep_df = sweep_df.set_index(SWEEP_PARAM)
display_cols = ['final_value', 'total_return', 'sharpe', 'max_drawdown', 'total_trades', 'win_rate']
sweep_df[[c for c in display_cols if c in sweep_df.columns]]

,final_value,total_return,sharpe,max_drawdown,total_trades,win_rate
rsi_oversold,,,,,,
25,51857.258543,0.036472,None,0.025713,4,0.25
30,51851.552983,0.036362,None,0.025713,4,0.25
35,51857.914355,0.036485,None,0.025713,4,0.25
40,52450.395978,0.047845,None,0.025713,4,0.50


In [7]:
# Cell 7: Swap strategy
# Uncomment and modify to test a custom strategy:
#
# from strategy.my_strategy import MyStrategy
#
# feeds = {pair: fetch_binance_feed(pair, days=DAYS) for pair in PAIRS}
# cerebro = build_cerebro(
#     feeds=feeds,
#     strategy_class=MyStrategy,
#     strategy_config={'my_threshold': 0.5},
#     fear_greed_map=fng,
# )
# results = cerebro.run()
# cerebro.plot(style='candlestick', volume=False)
# extract_metrics(results[0])

In [8]:
# Cell 8: Export results
import os
os.makedirs('../results', exist_ok=True)

# Trade log
if strat.trade_log:
    trade_df = pd.DataFrame(strat.trade_log)
    trade_df.to_csv('../results/trades.csv', index=False)
    print(f'Saved {len(trade_df)} round-trip trades to results/trades.csv')
    display(trade_df)
else:
    print('No closed trades to export.')

# Save plot
fig = cerebro.plot(style='candlestick', volume=False)[0][0]
fig.savefig('../results/backtest_chart.png', dpi=150, bbox_inches='tight')
print('Chart saved to results/backtest_chart.png')

Saved 3 round-trip trades to results/trades.csv


,pair,pnl,pnlcomm,baropen,barclose
0,ETH/USD,-365.070489,-371.541287,227,284
1,BTC/USD,-137.867569,-144.402529,423,447
2,SOL/USD,474.485560,461.013780,624,741


<IPython.core.display.Javascript object>

Chart saved to results/backtest_chart.png
